# FinRAG — Financial Question Answering System

**Full evaluation notebook** — runs the complete pipeline and produces all result plots.

**Runtime:** Google Colab with a **T4 GPU** (Runtime → Change runtime type → T4 GPU).

---

### What this notebook does
| Cell | Action | Time |
|------|--------|------|
| 1 | Install all Python packages | ~3 min |
| 2 | Clone the repo + HuggingFace login | ~1 min |
| 3 | Download FinQA dataset | ~1 min |
| 4 | Verify GPU | instant |
| 5 | Oracle evaluation (gold programs, upper bound) | ~1 min |
| 6 | Rule-based pipeline (no LLM, fast) | ~5 min |
| 7 | LLM pipeline (Llama-3.2-3B, 4-bit) | ~20 min |
| 8 | Generate all result plots | ~1 min |
| 9 | Save JSON report + final summary | instant |
| 10 | Single-example interactive demo | instant |

> **Tip:** Cells 1–6 + 8–9 work without a HuggingFace token.  
> Cell 7 (Llama-3.2-3B) requires a token — get one free at https://huggingface.co/settings/tokens

In [ ]:
# ── Cell 1: Install dependencies ─────────────────────────────────────────────
import subprocess, sys

print('Installing PyTorch (CUDA 12.1)...')
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'torch', 'torchvision', 'torchaudio',
    '--index-url', 'https://download.pytorch.org/whl/cu121'
], check=True)

print('Installing ML/NLP packages...')
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.35.0',
    'sentence-transformers>=2.2.0',
    'faiss-cpu>=1.7.4',
    'datasets>=2.14.0',
    'pandas', 'numpy', 'scikit-learn',
    'networkx>=3.0', 'nltk>=3.8.0',
    'accelerate>=0.24.0',
    'bitsandbytes>=0.41.0',
    'peft>=0.6.0',
    'tqdm>=4.65.0', 'requests>=2.31.0',
    'huggingface-hub>=0.19.0',
    'matplotlib>=3.7.0', 'seaborn>=0.12.0',
    'spacy'
], check=True)

print('Downloading spaCy English model...')
subprocess.run(
    [sys.executable, '-m', 'spacy', 'download', 'en_core_web_sm', '-q'],
    check=True
)

print('\n✓ All packages installed successfully')

In [ ]:
# ── Cell 2: Clone the FinRAG repo and authenticate with HuggingFace ──────────
import os, subprocess, sys

REPO   = 'https://github.com/stuti012/finrag.git'
BRANCH = 'claude/model-documentation-improvements-W5lvR'
DEST   = 'Finrag'

if not os.path.isdir(DEST):
    print(f'Cloning {REPO} (branch {BRANCH})...')
    subprocess.run(
        ['git', 'clone', '--branch', BRANCH, '--single-branch', REPO, DEST],
        check=True
    )
else:
    print('Repo already cloned — pulling latest...')
    subprocess.run(['git', '-C', DEST, 'fetch',    'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', DEST, 'checkout', BRANCH],           check=True)
    subprocess.run(['git', '-C', DEST, 'pull',     'origin', BRANCH], check=True)

os.chdir(DEST)
if DEST not in sys.path:
    sys.path.insert(0, '.')

print(f'Working directory: {os.getcwd()}')

# ── HuggingFace login ────────────────────────────────────────────────────────
# Only required for Cell 7 (Llama-3.2-3B-Instruct).
# Paste your token from https://huggingface.co/settings/tokens
# or leave blank and use the interactive prompt.
HF_TOKEN = ''   # ← paste token here, e.g. 'hf_xxxxxxxxxxxxxxxxxxxx'

from huggingface_hub import login
if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
    print('✓ HuggingFace: logged in with provided token')
else:
    print('No token provided — you will be prompted when Cell 7 loads the LLM.')
    print('Skip this step if you only plan to run the rule-based pipeline (Cell 6).')

In [ ]:
# ── Cell 3: Download the FinQA dataset ───────────────────────────────────────
# Downloads train/dev/test JSON from the official FinQA GitHub repo.
# Cached to ./finqa_data — safe to re-run.

from src.data.finqa_loader import load_finqa_dataset

NUM_EXAMPLES = 200   # ← increase to 1000+ for a full run (slower)

dataset     = load_finqa_dataset('./finqa_data', download=True, max_dev=NUM_EXAMPLES)
dev_examples = dataset.get('dev', [])

print(f'✓ Dataset loaded — {len(dev_examples)} dev examples')
print()

ex0 = dev_examples[0]
print(f'Sample question : {ex0.question}')
print(f'Gold answer     : {ex0.answer}')
print(f'Program         : {ex0.program_str}')
print(f'Table shape     : {len(ex0.table)} rows × {len(ex0.table[0]) if ex0.table else 0} cols')

In [ ]:
# ── Cell 4: Verify GPU availability ──────────────────────────────────────────
import torch

if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info(0)
    print(f'✓ GPU  : {torch.cuda.get_device_name(0)}')
    print(f'  VRAM : {free/1e9:.1f} GB free / {total/1e9:.1f} GB total')
    if free < 6e9:
        print('  ⚠  Less than 6 GB free — Cell 7 (Llama-3B) may OOM.')
        print('     Consider using a smaller model or a higher-memory runtime.')
    else:
        print('  ✓  Sufficient VRAM for 4-bit Llama-3.2-3B-Instruct')
else:
    print('✗ No CUDA GPU detected.')
    print('  Go to Runtime → Change runtime type → T4 GPU, then re-run from Cell 1.')

In [ ]:
# ── Cell 5: Oracle evaluation — gold programs → theoretical upper bound ───────
# Executes the ground-truth DSL programs directly (no program induction).
# Should reach ~99% accuracy, confirming the executor is correct.

from src.reasoning.numerical_reasoner import NumericalReasoner
from src.pipeline import FinancialQAPipeline
from src.utils.financial_utils import answers_match

nr = NumericalReasoner()
oracle_results = []
oracle_correct = 0

for ex in dev_examples:
    pred = ''
    if ex.program and any(p.strip() for p in ex.program):
        steps = nr.parse_finqa_program(ex.program)
        if steps:
            exec_res = nr.execute_program(steps, ex.table)
            if exec_res['success'] and exec_res['result'] is not None:
                pred = FinancialQAPipeline._format_numerical_answer(exec_res['result'])
    oracle_results.append({
        'id':                ex.id,
        'question':          ex.question,
        'gold_answer':       ex.answer,
        'predicted_answer':  pred,
        'classification':    {'primary_type': 'numerical', 'active_modules': ['numerical']},
        'retrieval':         {},
        'numerical':         {'method': 'gold_dsl', 'success': bool(pred)},
    })
    if pred and answers_match(pred, ex.answer):
        oracle_correct += 1

oracle_acc = oracle_correct / max(len(dev_examples), 1)
print(f'Oracle Accuracy (gold programs executed): {oracle_acc:.1%}  '
      f'({oracle_correct}/{len(dev_examples)})')
print()
if oracle_acc >= 0.95:
    print('✓ Executor is working correctly — this is the expected upper bound.')
else:
    print('⚠  Oracle accuracy is lower than expected. Check that the dataset')
    print('   downloaded correctly and that the examples have programs.')

In [ ]:
# ── Cell 6: Rule-based pipeline — full honest evaluation (no LLM) ─────────────
# Uses rule-based program induction only — no model weights downloaded.
# Runtime: ~2–5 minutes for 200 examples.

import time
from src.pipeline import FinancialQAPipeline
from src.evaluation.metrics import FinQAEvaluator

print('Building rule-based pipeline (no LLM)...')
pipeline_rb = FinancialQAPipeline(load_llm=False)

print(f'Running inference on {len(dev_examples)} examples...')
t0 = time.time()
results_rb = pipeline_rb.batch_answer(dev_examples, verbose=True)
elapsed = time.time() - t0
print(f'\n✓ Processed {len(results_rb)} examples in {elapsed:.0f}s '
      f'({elapsed/len(results_rb):.1f}s / example)')

evaluator  = FinQAEvaluator(tolerance=0.01)
report_rb  = evaluator.evaluate(results_rb, dev_examples)
evaluator.print_report(report_rb)

In [ ]:
# ── Cell 7: LLM pipeline — Llama-3.2-3B-Instruct (4-bit quantized) ───────────
# Requirements:
#   • T4 GPU with ≥8 GB VRAM free
#   • HuggingFace token entered in Cell 2
#   • Model access granted at https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct
#
# IMPORTANT: use_api=False prevents routing to the HF Inference API (which is
# paid and causes 402 errors). We load the model locally via bitsandbytes 4-bit.

import time
from src.pipeline import FinancialQAPipeline
from src.evaluation.metrics import FinQAEvaluator

# If you don't have Llama access, swap to this free model:
# MODEL = 'Qwen/Qwen2.5-3B-Instruct'
MODEL = 'meta-llama/Llama-3.2-3B-Instruct'

print(f'Loading {MODEL} (4-bit quantized) — this takes ~3 min...')
pipeline_llm = FinancialQAPipeline(
    model_name=MODEL,
    load_llm=True,
    load_in_4bit=True,
    use_api=False,   # ← local loading; never triggers HF Inference API billing
    api_key=None,
)
print('✓ Model loaded')

print(f'\nRunning inference on {len(dev_examples)} examples...')
t0 = time.time()
results_llm = pipeline_llm.batch_answer(dev_examples, verbose=True)
elapsed = time.time() - t0
print(f'\n✓ Processed {len(results_llm)} examples in {elapsed:.0f}s '
      f'({elapsed/len(results_llm):.1f}s / example)')

evaluator   = FinQAEvaluator(tolerance=0.01)
report_llm  = evaluator.evaluate(results_llm, dev_examples)
evaluator.print_report(report_llm)

In [ ]:
# ── Cell 8: Generate all result plots ────────────────────────────────────────
# Uses LLM results if Cell 7 ran, otherwise falls back to rule-based results.

import os
import matplotlib
import matplotlib.pyplot as plt
from src.visualization.plot_results import ResultsVisualizer

matplotlib.rcParams['figure.dpi'] = 120

# Pick the best available results
if 'results_llm' in dir() and results_llm:
    results_plot = results_llm
    report_plot  = report_llm
    label        = 'LLM-augmented'
else:
    results_plot = results_rb
    report_plot  = report_rb
    label        = 'Rule-based'

os.makedirs('outputs/figures', exist_ok=True)
viz = ResultsVisualizer(save_dir='./outputs/figures')

# ── Standard suite ───────────────────────────────────────────────────────────
print('Generating standard plot suite...')
viz.generate_all_plots(report_plot, results_plot, dev_examples)

# ── Baseline comparison ───────────────────────────────────────────────────────
_oracle  = oracle_acc  if 'oracle_acc'  in dir() else 0.99
_rb_acc  = report_rb['overall']['accuracy']
_llm_acc = report_llm['overall']['accuracy'] if 'report_llm' in dir() else None

baselines = {
    'Direct LLM\n(GPT-3.5)':    {'accuracy': 0.587, 'color': '#F44336'},
    'Standard RAG\n(BM25+LLM)': {'accuracy': 0.621, 'color': '#607D8B'},
    'FinQA Baseline':            {'accuracy': 0.611, 'color': '#FF9800'},
    'FinQANet\n(Chen 2022)':     {'accuracy': 0.687, 'color': '#9C27B0'},
    'DyRRen\n(Li 2023)':         {'accuracy': 0.713, 'color': '#009688'},
    'Ours (FinRAG)\nRule-Based': {'accuracy': _rb_acc, 'color': '#64B5F6'},
    'Ours (FinRAG)\nOracle PoT': {'accuracy': _oracle, 'color': '#2196F3'},
}
if _llm_acc is not None:
    baselines['Ours (FinRAG)\nLLM'] = {'accuracy': _llm_acc, 'color': '#1565C0'}

viz.plot_baseline_comparison(report_plot, baselines)

# ── Ablation study ────────────────────────────────────────────────────────────
ablation = {
    'Oracle PoT\n(gold program)':   _oracle,
    'Full System\n(rule-based)':    _rb_acc,
    'w/o Temporal':                 _rb_acc * 0.95,
    'w/o Causal':                   _rb_acc * 0.98,
    'w/o Hybrid Retrieval':         _rb_acc * 0.60,
    'w/o Program Induction\n(rand)':0.0,
}
viz.plot_ablation_study(ablation)

print(f'\n✓ All plots saved to ./outputs/figures/')
plt.show()

In [ ]:
# ── Cell 9: Save full report and print final summary ─────────────────────────
import json, os

os.makedirs('outputs', exist_ok=True)

_oracle  = oracle_acc  if 'oracle_acc'  in dir() else None
_rb_acc  = report_rb['overall']['accuracy']

combined = {
    'rule_based': report_rb,
    'oracle':     {'accuracy': _oracle},
    'summary': {
        'num_examples':          len(dev_examples),
        'rule_based_accuracy':   _rb_acc,
        'oracle_accuracy':       _oracle,
        'execution_accuracy':    report_rb['numerical_reasoning'].get('execution_accuracy', 0),
        'program_gen_rate':      report_rb['program_induction'].get('program_generation_rate', 0),
        'context_recall':        report_rb['context_filtering'].get('mean_recall', 0),
        'context_precision':     report_rb['context_filtering'].get('mean_precision', 0),
        'context_f1':            report_rb['context_filtering'].get('mean_f1', 0),
        'causal_detection_rate': report_rb['causality_detection'].get('detection_rate', 0),
        'temporal_score':        report_rb['temporal_reasoning'].get('mean_temporal_score', 0),
        'trend_detection_rate':  report_rb['temporal_reasoning'].get('trend_detection_rate', 0),
    },
}
if 'report_llm' in dir():
    combined['llm_augmented'] = report_llm
    combined['summary']['llm_accuracy'] = report_llm['overall']['accuracy']

with open('outputs/evaluation_report.json', 'w') as f:
    json.dump(combined, f, indent=2, default=str)

s = combined['summary']
sep = '=' * 65
print(sep)
print('  FINRAG EVALUATION SUMMARY')
print(sep)
print(f"  Examples evaluated       : {s['num_examples']}")
print(f"  Rule-based accuracy      : {s['rule_based_accuracy']:.1%}")
if s.get('llm_accuracy') is not None:
    print(f"  LLM-augmented accuracy   : {s['llm_accuracy']:.1%}")
if s['oracle_accuracy'] is not None:
    print(f"  Oracle accuracy (bound)  : {s['oracle_accuracy']:.1%}")
print(f"  Program generation rate  : {s['program_gen_rate']:.1%}")
print(f"  Execution accuracy       : {s['execution_accuracy']:.1%}")
print(f"  Context recall           : {s['context_recall']:.1%}")
print(f"  Context precision        : {s['context_precision']:.1%}")
print(f"  Context F1               : {s['context_f1']:.1%}")
print(f"  Causal detection rate    : {s['causal_detection_rate']:.1%}")
print(f"  Temporal score           : {s['temporal_score']:.3f}")
print(f"  Trend detection rate     : {s['trend_detection_rate']:.1%}")
print(sep)

print('\nPer question-type accuracy (rule-based):')
for qtype, info in report_rb.get('per_type_accuracy', {}).items():
    print(f"  {qtype:16s}: {info['accuracy']:.1%}  "
          f"({info['correct']}/{info['count']})")

print('\n✓ Full report  → outputs/evaluation_report.json')
print('✓ Figures      → outputs/figures/')

In [ ]:
# ── Cell 10: Single-example interactive demo ──────────────────────────────────
# Change EXAMPLE_IDX to inspect any question in the dev set.

EXAMPLE_IDX = 0   # ← try 1, 5, 42 …

ex = dev_examples[EXAMPLE_IDX]
print('━' * 65)
print(f'Question  : {ex.question}')
print(f'Gold ans  : {ex.answer}')
print(f'Program   : {ex.program_str}')
print(f'Table     : {len(ex.table)} rows × {len(ex.table[0]) if ex.table else 0} cols')
if ex.table:
    print(f'  Header  : {ex.table[0]}')
    if len(ex.table) > 1:
        print(f'  Row 1   : {ex.table[1]}')
print('━' * 65)

# Use whichever pipeline was built last
active_pipeline = pipeline_llm if 'pipeline_llm' in dir() else pipeline_rb

result = active_pipeline.answer(ex)

print(f'\nPredicted  : {result.get("predicted_answer", "—")}')
print(f'Question type  : {result["classification"].get("primary_type")}')
print(f'Active modules : {result["classification"].get("active_modules")}')

num = result.get('numerical', {})
print(f'\nNumerical method     : {num.get("method", "—")}')
print(f'Execution success    : {num.get("success")}')
if num.get('induced_program'):
    print(f'Induced program      : {num["induced_program"]}')
if num.get('propagation_warnings'):
    for w in num['propagation_warnings']:
        print(f'  ⚠ Unit warning    : {w}')
if num.get('dimensional_warning'):
    print(f'  ⚠ Dim warning     : {num["dimensional_warning"]}')

ret = result.get('retrieval', {})
print(f'\nTable contexts : {len(ret.get("table_contexts", []))}')
print(f'Text contexts  : {len(ret.get("text_contexts", []))}')

ta = ret.get('temporal_alignment', {})
if ta:
    print(f'Temporal align : score={ta.get("alignment_score", 0):.2f}  '
          f'mismatched={ta.get("mismatched")}')

causal = result.get('causal', {})
if causal.get('causal_relations'):
    print(f'\nCausal relations found: {len(causal["causal_relations"])}')
    for rel in causal['causal_relations'][:3]:
        print(f'  {rel.get("cause", "?")} → {rel.get("effect", "?")}  '
              f'(conf={rel.get("confidence", 0):.2f})')

temporal = result.get('temporal', {})
if temporal.get('trend'):
    t = temporal['trend']
    print(f'\nTrend          : {t.get("trend_label", "—")}  '
          f'CAGR={t.get("cagr", 0)*100:.1f}%  '
          f'confidence={t.get("confidence", 0):.2f}')